# YOLO Classification Example

## YOLO Image Classification

This notebook is an example of image classification with **Ultralytics YOLO**.

```
Image (URL or file) → YOLO classifier → Top-5 predictions with confidence scores
```

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x-cls.pt` | 28.4 M | slowest / most accurate |
| `yolo11l-cls.pt` | 12.9 M | — |
| `yolo11m-cls.pt` | 10.4 M | — |
| `yolo11s-cls.pt` |  5.5 M | — |
| `yolo11n-cls.pt` |  1.5 M | fastest / lightest |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/Practical-ML/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

In [ ]:
# Download sample images from GitHub
import os

if not os.path.exists(f'{PROJECT_PATH}/images'):
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/mastnk/practical-ml /tmp/practical-ml -q
    !cd /tmp/practical-ml && git sparse-checkout set yolo/images
    !cp -r /tmp/practical-ml/yolo/images "{PROJECT_PATH}/images"
    print('Images downloaded.')
else:
    print('Images already exist, skipping.')

%cd "{PROJECT_PATH}"
!ls images/

## Using Your Own Images

There are two ways to provide images for classification:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
!python classification.py --url https://example.com/image.jpg
```

**② Upload images to the `images/` folder**  
Place your image files in `PROJECT_PATH/images/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / Practical-ML / yolo / images/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
!python classification.py --file images/your_image.jpg   # single file
!python classification.py --dir  images/                 # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'yolo11x-cls.pt'  # 28.4M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-cls.pt'  # 12.9M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 10.4M params
# MODEL_NAME = 'yolo11s-cls.pt'  #  5.5M params
MODEL_NAME   = 'yolo11n-cls.pt'  #  1.5M params — fastest, lightest  ← active
```

Larger models are more accurate but take longer to run. Start with `yolo11n-cls.pt` for quick experiments.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile classification.py
"""YOLO Image Classification — command-line interface.

Usage:
  %run classification.py --url  <image_url>  [--disp] [--top_k N]
  %run classification.py --file <image_path> [--disp] [--top_k N]
  %run classification.py --dir  <image_dir>  [--disp] [--top_k N]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-cls.pt'  #  1.5M params — fastest / lightest
# MODEL_NAME = 'yolo11s-cls.pt'  #  5.5M params
# MODEL_NAME = 'yolo11m-cls.pt'  # 10.4M params
# MODEL_NAME = 'yolo11l-cls.pt'  # 12.9M params
# MODEL_NAME = 'yolo11x-cls.pt'  # 28.4M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/Practical-ML/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(image, label: str, disp: bool) -> None:
    """When --disp is active, display the image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(image)
        print(label)

# ---- Functions ---------------------------------------------------
def classify_url(url: str, top_k: int = 5, disp: bool = False):
    """Download an image from a URL and classify it."""
    image   = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    show(image, url, disp)
    results = model.predict(image, verbose=False)
    probs   = results[0].probs
    names   = results[0].names
    pairs   = [(names[idx], float(conf))
               for idx, conf in zip(probs.top5[:top_k], probs.top5conf[:top_k])]
    for i, (name, conf) in enumerate(pairs, 1):
        print(f'{i:2d}: {name:<25} {conf*100:.1f}%')
    return pairs


def classify_file(path: str, top_k: int = 5, disp: bool = False):
    """Classify a single local image file."""
    image   = Image.open(path).convert('RGB')
    show(image, path, disp)
    results = model.predict(image, verbose=False)
    probs   = results[0].probs
    names   = results[0].names
    pairs   = [(names[idx], float(conf))
               for idx, conf in zip(probs.top5[:top_k], probs.top5conf[:top_k])]
    for i, (name, conf) in enumerate(pairs, 1):
        print(f'{i:2d}: {name:<25} {conf*100:.1f}%')
    return pairs


def classify_dir(directory: str, top_k: int = 5, disp: bool = False):
    """Classify all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        classify_file(path, top_k, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Image Classification')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',   type=str,            help='Image URL to classify')
group.add_argument('--file',  type=str,            help='Path to a single image file')
group.add_argument('--dir',   type=str,            help='Directory of images to classify')
parser.add_argument('--top_k', type=int, default=5, help='Number of top predictions (default: 5)')
parser.add_argument('--disp',  action='store_true', help='Display image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    classify_url(args.url,   top_k=args.top_k, disp=args.disp)
elif args.file:
    classify_file(args.file, top_k=args.top_k, disp=args.disp)
elif args.dir:
    classify_dir(args.dir,   top_k=args.top_k, disp=args.disp)


## `classification.py` Usage

After running the `%%writefile` cell above, `classification.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run classification.py --url  <image_url>        # classify a remote image
%run classification.py --file <image_path>       # classify a single local file
%run classification.py --dir  <image_directory>  # classify all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Print filename / URL and display image before results |
| `--top_k <n>` | `5` | Number of top predictions to show |

**Examples**

```bash
# Classify a remote image and display it
%run classification.py --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg --disp

# Classify one file, show top-3 with image
%run classification.py --file images/cat.jpg --top_k 3 --disp

# Classify every image in the folder, display each one
%run classification.py --dir images --disp
```

**Output format (with `--disp`)**

```
<image displayed inline>
images/puppy.jpg
 1: golden_retriever          91.3%
 2: Labrador_retriever         5.1%
 3: kuvasz                     1.2%
```


## Execution Methods

Use **`%run`** to execute `classification.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and classifies a remote image |
| Single file | `--file <path>` | Classifies one local image |
| Directory | `--dir <path>` | Classifies all images in a folder |

Add `--disp` to display each image and its filename / URL before the results.  
Add `--top_k <n>` to change the number of predictions shown (default: 5).


In [ ]:
# Execute: classify an image from URL (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --url https://cdn.pixabay.com/photo/2016/12/13/05/15/puppy-1903313_640.jpg

In [ ]:
# Execute: classify a single local image file (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --file images/teefarm-pile-1651945_640.jpg

In [ ]:
# Execute: classify all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run classification.py --disp --dir images

💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.

[![Return to GitHub](https://img.shields.io/badge/GitHub-Return-black?logo=github)](https://github.com/mastnk/practical-ml)